# Player Statistics Data Preprocessing

This notebook performs the data preprocessing of scraped data for the model that predicts using player statistics. We merge and clean our raw data stored in `\data`.

The attributes I chose to use in building the model include:

**Points, Assists, Rebounds, Steals and Blocks**, all on a per game basis. The model would have been improved by using statistics that further distinguish positions such as three point percentage (as guards and forwards are usually much better than centers in this department), turnovers or free throw percentage.  

However, due to these more advanced statistics rarely being recorded outside of professional games, I decided to only use the traditionally recorded statistics of points, assists, rebounds, steals and blocks as the average user who may have only played up to high school basketball would either have these attributes recorded or know a rough estimate of their numbers for these attributes. For example, american highschool varsity basketball only records these statistics: [See Here](https://www.maxpreps.com/basketball/stat-leaders/).

## Imports

In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re

## (1) Merge Data

In [14]:
# Reading in data
player_statistics = pd.read_csv(os.path.join("data", "Seasons_Stats.csv"))
active_players = pd.read_csv(os.path.join("data", "active_players.csv"))
active_players_processed = pd.read_csv(os.path.join("data", "active_players_processed.csv"))
player_statistics_processed = pd.read_csv(os.path.join("data", "player_stats_processed.csv"))

## (2) Clean

Convert all column headers into lowercase to have a consistent case between all columns.

In [15]:
player_statistics.columns = player_statistics.columns.str.lower()

Keep only relevant attributes that are feasible (see explanation above) and may have a relationship with a players position (Year, Age).

In [16]:
player_statistics = player_statistics[['player', 'pos', 'pts', 'ast', 'trb', 'stl', 'blk', 'age', 'year', 'g']]
player_statistics.head()

,player,pos,pts,ast,trb,stl,blk,age,year,g
0,Curly Armstrong,G-F,458.0,176.0,NaN,NaN,NaN,31.0,1950.0,63.0
1,Cliff Barker,SG,279.0,109.0,NaN,NaN,NaN,29.0,1950.0,49.0
2,Leo Barnhorst,SF,438.0,140.0,NaN,NaN,NaN,25.0,1950.0,67.0
3,Ed Bartels,F,63.0,20.0,NaN,NaN,NaN,24.0,1950.0,15.0
4,Ed Bartels,F,59.0,20.0,NaN,NaN,NaN,24.0,1950.0,13.0


Drop all NA values as we're going to want to keep only players with all the relevant statistics. Furthermore, any duplicates entries will be dropped.

In [17]:
# Checking the shape, number of duplicates and missing values
print(f'The number of rows in the dataset is: {player_statistics.shape[0]}')
print(f'The number of columns/features in the dataset is: {player_statistics.shape[1]}')
print(f'The number of duplicate entries in the dataset is: {player_statistics.duplicated().sum()}')
print(f'The number of missing values in the dataset is: {player_statistics.isna().sum().sum()}')

The number of rows in the dataset is: 24691
The number of columns/features in the dataset is: 10
The number of duplicate entries in the dataset is: 67
The number of missing values in the dataset is: 8644


In [18]:
player_statistics = player_statistics.dropna()
player_statistics = player_statistics.drop_duplicates().reset_index(drop=True)

In [19]:
# Checking the number of duplicates and missing values
print(f'The number of duplicate entries in the dataset is: {player_statistics.duplicated().sum()}')
print(f'The number of missing values in the dataset is: {player_statistics.isna().sum().sum()}')

The number of duplicate entries in the dataset is: 0
The number of missing values in the dataset is: 0


It is evident that the statistics are shown as **totals**, and not their statistics **per game**. Although users could calculate their total statistics, it's more common that a players statistics are recorded as per game. Therefore, next we will convert all the statistics into per game.

In [20]:
# Convert stats into per game
stats_cols = ['pts', 'ast', 'trb', 'stl', 'blk']
for col in stats_cols:
    player_statistics[col] = np.round(player_statistics[col].div(player_statistics['g']), 2)
    
# Drop games
player_statistics.drop(columns=['g'], inplace=True)
player_statistics.head()

,player,pos,pts,ast,trb,stl,blk,age,year
0,Zaid Abdul-Aziz,C,10.95,2.10,11.68,1.01,1.32,27.0,1974.0
1,Kareem Abdul-Jabbar*,C,27.05,4.77,14.54,1.38,3.49,26.0,1974.0
2,Don Adams,SF,10.26,1.91,6.05,1.49,0.16,26.0,1974.0
3,Rick Adelman,PG,3.31,1.02,1.25,0.65,0.02,27.0,1974.0
4,Lucius Allen,PG,17.61,5.19,4.04,1.90,0.31,26.0,1974.0


It is evident that the Guard position is broken down into PG (Point Guard) and SG (Shooting Guard). Additionally, the Forward position is broken down into SF (Small Forward) and Power Forward (PF). 

Similar to before, we're trying to predict the general position of a player (Guard, Forward, Center) and so we handle this below.

In [21]:
player_statistics['pos'] = player_statistics['pos'].apply(lambda x: x[-1])
player_statistics.head()

,player,pos,pts,ast,trb,stl,blk,age,year
0,Zaid Abdul-Aziz,C,10.95,2.10,11.68,1.01,1.32,27.0,1974.0
1,Kareem Abdul-Jabbar*,C,27.05,4.77,14.54,1.38,3.49,26.0,1974.0
2,Don Adams,F,10.26,1.91,6.05,1.49,0.16,26.0,1974.0
3,Rick Adelman,G,3.31,1.02,1.25,0.65,0.02,27.0,1974.0
4,Lucius Allen,G,17.61,5.19,4.04,1.90,0.31,26.0,1974.0


Number of rows.

In [24]:
print(len(player_statistics))

20796


## (3) Export results

In [22]:
# Export player statistics processed
#player_statistics.to_csv(os.path.join("data", "player_stats_processed.csv"))

## Dataset 2:

In [93]:
# Retrieving relevant datasets
dates = ['20250728']
date_pattern = '|'.join(dates)
pattern = rf"nba_current_player_stats_(?:{date_pattern})_batch[0-9]+\.csv"

# Get list of datasets in \data which match the pattern
matching_files = [f for f in os.listdir("data") if re.match(pattern, f)]

print(matching_files)

# Read in and merge data
api_player_stats = pd.DataFrame()
for file in matching_files:
    df = pd.read_csv(os.path.join("data", file))
    api_player_stats = pd.concat([api_player_stats, df])

print(len(api_player_stats))
api_player_stats.head()

['nba_current_player_stats_20250728_batch1.csv', 'nba_current_player_stats_20250728_batch3.csv', 'nba_current_player_stats_20250728_batch2.csv']
2687


,player,pos,PTS,AST,REB,STL,BLK,TOV,AST_TO,STOCKS,FIC,age,year
0,Precious Achiuwa,F,4.98,0.48,3.41,0.33,0.46,0.70,0.685714,0.79,8.96,21.0,2021
1,Precious Achiuwa,F,9.10,1.12,6.48,0.51,0.56,1.15,0.973913,1.07,16.62,22.0,2022
2,Precious Achiuwa,F,9.24,0.91,5.96,0.56,0.55,1.07,0.850467,1.11,16.15,23.0,2023
3,Precious Achiuwa,F,7.72,1.76,5.44,0.64,0.48,1.16,1.517241,1.12,14.88,24.0,2024
4,Precious Achiuwa,F,7.59,1.08,7.16,0.61,1.14,1.10,0.981818,1.75,16.48,24.0,2024


In [94]:
print(f'The number of rows in the dataset is: {api_player_stats.shape[0]}')
print(f'The number of columns/features in the dataset is: {api_player_stats.shape[1]}')
print(f'The number of duplicate entries in the dataset is: {api_player_stats.duplicated().sum()}')
print(f'The number of missing values in the dataset is: {api_player_stats.isna().sum().sum()}')

The number of rows in the dataset is: 2687
The number of columns/features in the dataset is: 13
The number of duplicate entries in the dataset is: 0
The number of missing values in the dataset is: 0


Identify entries with `inf`. This happens with the AST/TO (Assist to Turnover) ratio when Turnover's is 0, resulting in a division by 0, creating infinity values.

In [95]:
print(api_player_stats[api_player_stats == np.inf].count())

player     0
pos        0
PTS        0
AST        0
REB        0
STL        0
BLK        0
TOV        0
AST_TO    20
STOCKS     0
FIC        0
age        0
year       0
dtype: int64


* There exists 20 records with an `inf` value for AST_TO.

Remove the 20 records with `inf`.

In [96]:
# Select only numeric columns
api_player_stats_numeric = api_player_stats.select_dtypes(include=[np.number])

# Find rows with any infinity values in numeric columns
api_player_stats[np.isinf(api_player_stats_numeric).any(axis=1)]

,player,pos,PTS,AST,REB,STL,BLK,TOV,AST_TO,STOCKS,FIC,age,year
266,Tony Bradley,C,0.89,0.11,1.22,0.00,0.00,0.0,inf,0.00,2.22,20.0,2018
438,Justin Champagnie,G,2.00,0.33,1.33,0.00,0.00,0.0,inf,0.00,3.66,22.0,2023
439,Justin Champagnie,G,2.50,1.50,2.00,0.50,0.00,0.0,inf,0.50,6.50,22.0,2023
440,Justin Champagnie,G,2.20,0.80,1.60,0.20,0.00,0.0,inf,0.20,4.80,22.0,2023
543,Seth Curry,G,0.00,0.50,1.00,0.00,0.00,0.0,inf,0.00,1.50,24.0,2015
642,PJ Dozier,G,3.17,0.83,2.83,0.33,0.00,0.0,inf,0.33,7.16,22.0,2019
750,Adam Flagler,G,1.50,2.00,0.00,0.00,0.00,0.0,inf,0.00,3.50,24.0,2024
817,Taj Gibson,F,4.50,0.50,2.25,0.25,0.25,0.0,inf,0.50,7.75,39.0,2024
973,Ron Harper Jr.,G,2.22,0.44,0.78,0.00,0.11,0.0,inf,0.11,3.55,23.0,2023
974,Ron Harper Jr.,G,0.00,1.00,0.00,0.00,0.00,0.0,inf,0.00,1.00,24.0,2024


In [97]:
# Remove these rows
api_player_stats = api_player_stats[~np.isinf(api_player_stats_numeric).any(axis=1)]

In [101]:
# Export api player stats
api_player_stats.to_csv(os.path.join("data", "api_player_stats.csv"))

## Dataset 3:

Concatenate datasets generated in batches

In [76]:
# Retrieving relevant datasets
dates = ['20250728']
date_pattern = '|'.join(dates)
pattern = rf"nba_player_stats_(?:{date_pattern})_batch[0-9]+\.csv"

# Get list of datasets in \data which match the pattern
matching_files = [f for f in os.listdir("data") if re.match(pattern, f)]

# Read in and merge data
api_player_stats_2 = pd.DataFrame()
for file in matching_files:
    df = pd.read_csv(os.path.join("data", file))
    api_player_stats_2 = pd.concat([api_player_stats_2, df])

print(len(api_player_stats_2))
api_player_stats_2.head()

6674


,player,pos,PTS,AST,REB,STL,BLK,TOV,AST_TO,STOCKS,FIC,age,year
0,Aaron Brooks,G,11.63,3.18,2.02,0.66,0.18,1.91,1.66,0.84,15.76,30.0,2015
1,Aaron Brooks,G,7.12,2.61,1.46,0.43,0.14,1.19,2.19,0.57,10.57,31.0,2016
2,Aaron Brooks,G,4.95,1.92,1.06,0.38,0.14,1.02,1.88,0.52,7.43,32.0,2017
3,Aaron Brooks,G,2.34,0.62,0.53,0.19,0.00,0.34,1.82,0.19,3.34,33.0,2018
4,Aaron Gordon,F,5.17,0.70,3.60,0.45,0.47,0.81,0.86,0.92,9.58,19.0,2015


In [77]:
print(f'The number of rows in the dataset is: {api_player_stats_2.shape[0]}')
print(f'The number of columns/features in the dataset is: {api_player_stats_2.shape[1]}')
print(f'The number of duplicate entries in the dataset is: {api_player_stats_2.duplicated().sum()}')
print(f'The number of missing values in the dataset is: {api_player_stats_2.isna().sum().sum()}')

The number of rows in the dataset is: 6674
The number of columns/features in the dataset is: 13
The number of duplicate entries in the dataset is: 0
The number of missing values in the dataset is: 2


Identify and remove missing entries

In [78]:
print(api_player_stats_2[api_player_stats_2.isna().any(axis=1)])
api_player_stats_2.dropna(inplace=True)

               player  pos   PTS   AST   REB   STL   BLK   TOV  AST_TO  \
430  Tyler Hansbrough  NaN  3.65  0.28  3.58  0.41  0.20  0.27    1.04   
431  Tyler Hansbrough  NaN  2.36  0.18  2.02  0.27  0.16  0.27    0.67   

     STOCKS   FIC   age  year  
430    0.61  7.85  29.0  2015  
431    0.43  4.72  30.0  2016  


Identify entries with `inf`. This happens with the AST/TO (Assist to Turnover) ratio when Turnover's is 0, resulting in a division by 0, creating infinity values.

In [81]:
print(api_player_stats_2[api_player_stats_2 == np.inf].count())

player      0
pos         0
PTS         0
AST         0
REB         0
STL         0
BLK         0
TOV         0
AST_TO    150
STOCKS      0
FIC         0
age         0
year        0
dtype: int64


* There exists 150 records with an `inf` value for AST_TO.

Remove the 150 records with `inf`.

In [83]:
# Select only numeric columns
api_player_stats_2_numeric = api_player_stats_2.select_dtypes(include=[np.number])

# Find rows with any infinity values in numeric columns
api_player_stats_2[np.isinf(api_player_stats_2_numeric).any(axis=1)]

,player,pos,PTS,AST,REB,STL,BLK,TOV,AST_TO,STOCKS,FIC,age,year
17,Aaron Harrison,G,0.20,0.60,0.60,0.00,0.00,0.0,inf,0.00,1.40,22.0,2017
87,Alex Stepheson,F,5.00,0.50,6.50,0.25,0.75,0.0,inf,1.00,13.00,28.0,2016
88,Alex Stepheson,F,2.75,0.25,3.50,0.12,0.62,0.0,inf,0.74,7.24,28.0,2016
191,Anthony Brown,F,3.00,1.00,0.00,0.00,0.00,0.0,inf,0.00,4.00,25.0,2018
205,Anthony Morrow,G,4.56,0.67,0.22,0.22,0.00,0.0,inf,0.22,5.67,31.0,2017
...,...,...,...,...,...,...,...,...,...,...,...,...,...
679,Didi Louzada,G,0.00,0.50,1.00,0.00,0.00,0.0,inf,0.00,1.50,22.0,2022
685,Dylan Windler,G,1.00,0.33,0.33,0.00,0.00,0.0,inf,0.00,1.66,27.0,2024
686,Dylan Windler,G,1.50,0.75,0.38,0.00,0.00,0.0,inf,0.00,2.63,27.0,2024
803,Malik Fitts,F,1.88,0.62,0.88,0.00,0.00,0.0,inf,0.00,3.38,24.0,2022


In [84]:
# Remove these rows
api_player_stats_2 = api_player_stats_2[~np.isinf(api_player_stats_2_numeric).any(axis=1)]

In [86]:
# Export api player stats
api_player_stats_2.to_csv(os.path.join("data", "api_player_stats2.csv"), index=False)